In [8]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from datetime import datetime

print("=" * 100)
print("DELIVERY GAP CALCULATION AND CLASSIFICATION")
print("=" * 100)

# ------------------------------------------------------------
# PROJECT PATH CONFIGURATION
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\hp\Analysis in Global Supply Chain Operations"
)

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cleaned_logistics_data.csv"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"

for directory in [PROCESSED_DIR, REPORT_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ANALYTICAL_FILE = (
    PROCESSED_DIR / "delivery_gap_analysis.csv"
)

CLASSIFICATION_REPORT = (
    REPORT_DIR / "delivery_classification_report.csv"
)

RISK_MISMATCH_REPORT = (
    REPORT_DIR / "risk_mismatch_report.csv"
)

STATUS_MISMATCH_REPORT = (
    REPORT_DIR / "delivery_status_mismatch_report.csv"
)

CANCELLED_REPORT = (
    REPORT_DIR / "cancelled_shipments.csv"
)

INELIGIBLE_REPORT = (
    REPORT_DIR / "ineligible_delivery_records.csv"
)

ZERO_SCHEDULED_REPORT = (
    REPORT_DIR / "zero_scheduled_duration_investigation.csv"
)

VALIDATION_REPORT = (
    REPORT_DIR / "delivery_gap_validation_report.csv"
)

SUMMARY_JSON = (
    REPORT_DIR / "delivery_gap_analysis_summary.json"
)

ACTUAL_COL = "Days for shipping (real)"
SCHEDULED_COL = "Days for shipment (scheduled)"
STATUS_COL = "Delivery Status"
RISK_COL = "Late_delivery_risk"
MODE_COL = "Shipping Mode"

# ------------------------------------------------------------
# LOAD AND INSPECT CLEANED DATASET
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("LOAD AND INSPECT CLEANED DATASET")
print("=" * 100)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found: {INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

original_row_count = len(df)
original_columns = len(df.columns)

print(f"Source file: {INPUT_FILE}")
print(f"Dataset rows: {original_row_count:,}")
print(f"Dataset columns: {original_columns:,}")

required_columns = [
    ACTUAL_COL,
    SCHEDULED_COL,
    STATUS_COL,
    RISK_COL,
    MODE_COL
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

print(f"Actual shipping duration column: {ACTUAL_COL}")
print(f"Scheduled shipping duration column: {SCHEDULED_COL}")

# ------------------------------------------------------------
# IDENTIFY AND VALIDATE SHIPPING DURATIONS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("IDENTIFY AND VALIDATE SHIPPING DURATIONS")
print("=" * 100)

df[ACTUAL_COL] = pd.to_numeric(
    df[ACTUAL_COL],
    errors="coerce"
)

df[SCHEDULED_COL] = pd.to_numeric(
    df[SCHEDULED_COL],
    errors="coerce"
)

actual_valid = (
    df[ACTUAL_COL].notna()
    & np.isfinite(df[ACTUAL_COL])
    & (df[ACTUAL_COL] >= 0)
)

scheduled_valid = (
    df[SCHEDULED_COL].notna()
    & np.isfinite(df[SCHEDULED_COL])
    & (df[SCHEDULED_COL] >= 0)
)

duration_valid = actual_valid & scheduled_valid

print(f"Valid actual durations: {actual_valid.sum():,}")
print(f"Invalid actual durations: {(~actual_valid).sum():,}")
print(f"Valid scheduled durations: {scheduled_valid.sum():,}")
print(f"Invalid scheduled durations: {(~scheduled_valid).sum():,}")
print(f"Records with both valid durations: {duration_valid.sum():,}")

# ------------------------------------------------------------
# CALCULATE DELIVERY DELAY GAP
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CALCULATE DELIVERY DELAY GAP")
print("=" * 100)

df["Delivery Delay Gap"] = (
    df[ACTUAL_COL] - df[SCHEDULED_COL]
)

df.loc[
    ~duration_valid,
    "Delivery Delay Gap"
] = np.nan

df["Delivery Gap Validation"] = np.where(
    duration_valid,
    "Valid",
    "Invalid or Missing Duration"
)

independent_gap = (
    df.loc[duration_valid, ACTUAL_COL]
    - df.loc[duration_valid, SCHEDULED_COL]
)

gap_calculation_pass = np.allclose(
    df.loc[
        duration_valid,
        "Delivery Delay Gap"
    ].to_numpy(),
    independent_gap.to_numpy()
)

valid_gaps = df.loc[
    duration_valid,
    "Delivery Delay Gap"
]

print("Formula: Actual Shipping Days - Scheduled Shipping Days")
print(f"Calculated valid gaps: {len(valid_gaps):,}")

if len(valid_gaps) > 0:
    print(f"Minimum delivery gap: {valid_gaps.min():.2f} days")
    print(f"Maximum delivery gap: {valid_gaps.max():.2f} days")
    print(f"Average delivery gap: {valid_gaps.mean():.2f} days")
    print(f"Median delivery gap: {valid_gaps.median():.2f} days")

print(
    f"Gap calculation validation: "
    f"{'PASS' if gap_calculation_pass else 'FAIL'}"
)

# ------------------------------------------------------------
# INVESTIGATE ZERO SCHEDULED DURATION RECORDS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("INVESTIGATE ZERO SCHEDULED DURATION RECORDS")
print("=" * 100)

zero_scheduled = (
    scheduled_valid
    & df[SCHEDULED_COL].eq(0)
)

zero_scheduled_records = df.loc[
    zero_scheduled
].copy()

if not zero_scheduled_records.empty:

    zero_scheduled_records["Zero Scheduled Investigation"] = np.where(
        zero_scheduled_records[MODE_COL]
        .astype("string")
        .str.strip()
        .str.casefold()
        .eq("same day"),
        "Same Day shipment - retained",
        "Zero scheduled duration - investigate"
    )

    zero_mode_summary = (
        zero_scheduled_records[MODE_COL]
        .fillna("Missing")
        .value_counts(dropna=False)
        .rename_axis("Shipping Mode")
        .reset_index(name="Record Count")
    )

    zero_scheduled_records.to_csv(
        ZERO_SCHEDULED_REPORT,
        index=False
    )

    same_day_zero_count = int(
        zero_scheduled_records[MODE_COL]
        .astype("string")
        .str.strip()
        .str.casefold()
        .eq("same day")
        .sum()
    )

    print(
        f"Zero scheduled duration records: "
        f"{len(zero_scheduled_records):,}"
    )

    print("\nZero scheduled duration by shipping mode:")
    print(zero_mode_summary.to_string(index=False))

    print(
        f"\nZero-duration Same Day records: "
        f"{same_day_zero_count:,}"
    )

    print(
        f"Zero-duration non-Same Day records: "
        f"{len(zero_scheduled_records) - same_day_zero_count:,}"
    )

else:
    same_day_zero_count = 0

    zero_mode_summary = pd.DataFrame(
        columns=["Shipping Mode", "Record Count"]
    )

    print("No zero scheduled duration records found.")

    zero_scheduled_records.to_csv(
        ZERO_SCHEDULED_REPORT,
        index=False
    )

# ------------------------------------------------------------
# IDENTIFY CANCELLED SHIPMENTS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("IDENTIFY CANCELLED SHIPMENTS")
print("=" * 100)

df[STATUS_COL] = (
    df[STATUS_COL]
    .astype("string")
    .str.strip()
)

cancelled_mask = (
    df[STATUS_COL]
    .str.casefold()
    .eq("shipping canceled")
    .fillna(False)
)

cancelled_records = df.loc[
    cancelled_mask
].copy()

cancelled_records.to_csv(
    CANCELLED_REPORT,
    index=False
)

print(f"Cancelled shipments: {cancelled_mask.sum():,}")

# ------------------------------------------------------------
# DEFINE ELIGIBLE AND INELIGIBLE RECORDS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("DEFINE ELIGIBLE AND INELIGIBLE RECORDS")
print("=" * 100)

# Cancelled shipments are a separate category.
# Ineligible records refer only to non-cancelled records
# with invalid or missing actual/scheduled durations.

eligible_mask = (
    ~cancelled_mask
    & duration_valid
)

ineligible_mask = (
    ~cancelled_mask
    & ~duration_valid
)

print(f"Total records: {len(df):,}")
print(f"Eligible records: {eligible_mask.sum():,}")
print(f"Cancelled shipments: {cancelled_mask.sum():,}")
print(f"Invalid-duration records: {ineligible_mask.sum():,}")

# ------------------------------------------------------------
# CLASSIFY DELAYED, ON-TIME AND EARLY DELIVERIES
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CLASSIFY DELAYED, ON-TIME AND EARLY DELIVERIES")
print("=" * 100)

df["Delivery Classification"] = "Not Classified"

gap = df["Delivery Delay Gap"]

df.loc[
    eligible_mask & (gap > 0),
    "Delivery Classification"
] = "Delayed"

df.loc[
    eligible_mask & (gap == 0),
    "Delivery Classification"
] = "On-time"

df.loc[
    eligible_mask & (gap < 0),
    "Delivery Classification"
] = "Early"

df.loc[
    cancelled_mask,
    "Delivery Classification"
] = "Cancelled"

df.loc[
    ineligible_mask,
    "Delivery Classification"
] = "Ineligible"

delayed_count = int(
    (df["Delivery Classification"] == "Delayed").sum()
)

on_time_count = int(
    (df["Delivery Classification"] == "On-time").sum()
)

early_count = int(
    (df["Delivery Classification"] == "Early").sum()
)

cancelled_count = int(
    (df["Delivery Classification"] == "Cancelled").sum()
)

ineligible_count = int(
    (df["Delivery Classification"] == "Ineligible").sum()
)

eligible_count = int(eligible_mask.sum())

print(f"Eligible records: {eligible_count:,}")
print(f"Delayed deliveries: {delayed_count:,}")
print(f"On-time deliveries: {on_time_count:,}")
print(f"Early deliveries: {early_count:,}")
print(f"Cancelled shipments: {cancelled_count:,}")
print(f"Ineligible records: {ineligible_count:,}")

# ------------------------------------------------------------
# COMPARE CLASSIFICATION WITH DELIVERY STATUS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("COMPARE CLASSIFICATION WITH DELIVERY STATUS")
print("=" * 100)

status_normalized = (
    df[STATUS_COL]
    .astype("string")
    .str.strip()
    .str.casefold()
)

status_mapping = {
    "late delivery": "Delayed",
    "shipping on time": "On-time",
    "advance shipping": "Early",
    "shipping canceled": "Cancelled"
}

df["Delivery Status Classification"] = (
    status_normalized
    .map(status_mapping)
    .fillna("Unmapped Status")
)

df["Delivery Status Comparison"] = "Not Compared"

df.loc[
    eligible_mask
    & df["Delivery Status Classification"].eq(
        df["Delivery Classification"]
    ),
    "Delivery Status Comparison"
] = "Match"

df.loc[
    eligible_mask
    & ~df["Delivery Status Classification"].eq(
        df["Delivery Classification"]
    ),
    "Delivery Status Comparison"
] = "Mismatch"

status_mismatch_mask = (
    eligible_mask
    & df["Delivery Status Comparison"].eq("Mismatch")
)

status_mismatch_records = df.loc[
    status_mismatch_mask
].copy()

status_mismatch_records.to_csv(
    STATUS_MISMATCH_REPORT,
    index=False
)

status_comparison_summary = (
    df.loc[
        eligible_mask,
        "Delivery Status Comparison"
    ]
    .value_counts()
    .rename_axis("Comparison Result")
    .reset_index(name="Record Count")
)

print(f"Eligible records compared: {eligible_count:,}")
print(
    f"Matching records: "
    f"{(df['Delivery Status Comparison'] == 'Match').sum():,}"
)
print(f"Mismatched records: {status_mismatch_mask.sum():,}")

print("\nDelivery Status comparison:")
print(status_comparison_summary.to_string(index=False))

# ------------------------------------------------------------
# COMPARE CLASSIFICATION WITH LATE DELIVERY RISK
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("COMPARE CLASSIFICATION WITH LATE DELIVERY RISK")
print("=" * 100)

df[RISK_COL] = pd.to_numeric(
    df[RISK_COL],
    errors="coerce"
)

risk_valid = (
    df[RISK_COL].notna()
    & df[RISK_COL].isin([0, 1])
)

risk_comparison_mask = (
    eligible_mask
    & risk_valid
)

df["Late Delivery Risk Interpretation"] = np.select(
    [
        df[RISK_COL].eq(1),
        df[RISK_COL].eq(0)
    ],
    [
        "Risk Flagged",
        "Risk Not Flagged"
    ],
    default="Invalid or Missing Risk"
)

df["Late Delivery Risk Comparison"] = "Not Compared"

risk_mismatch_mask = (
    risk_comparison_mask
    & (
        (
            df["Delivery Classification"].eq("Delayed")
            & df[RISK_COL].ne(1)
        )
        |
        (
            df["Delivery Classification"].isin(
                ["On-time", "Early"]
            )
            & df[RISK_COL].ne(0)
        )
    )
)

df.loc[
    risk_comparison_mask,
    "Late Delivery Risk Comparison"
] = "Consistent"

df.loc[
    risk_mismatch_mask,
    "Late Delivery Risk Comparison"
] = "Mismatch"

risk_mismatch_records = df.loc[
    risk_mismatch_mask
].copy()

risk_mismatch_records.to_csv(
    RISK_MISMATCH_REPORT,
    index=False
)

risk_invalid_records = df.loc[
    eligible_mask & ~risk_valid
].copy()

risk_comparison_summary = (
    df.loc[
        risk_comparison_mask,
        "Late Delivery Risk Comparison"
    ]
    .value_counts()
    .rename_axis("Risk Comparison Result")
    .reset_index(name="Record Count")
)

print(
    f"Eligible records with valid risk labels: "
    f"{risk_comparison_mask.sum():,}"
)
print(f"Risk mismatches: {risk_mismatch_mask.sum():,}")
print(
    f"Eligible records with invalid/missing risk: "
    f"{len(risk_invalid_records):,}"
)

print("\nRisk comparison:")
print(risk_comparison_summary.to_string(index=False))

# ------------------------------------------------------------
# GENERATE INELIGIBLE RECORDS REPORT
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("GENERATE INELIGIBLE RECORDS REPORT")
print("=" * 100)

ineligible_records = df.loc[
    ineligible_mask
].copy()

ineligible_records["Ineligibility Reason"] = np.select(
    [
        ~actual_valid.loc[ineligible_records.index],
        ~scheduled_valid.loc[ineligible_records.index]
    ],
    [
        "Invalid or missing actual shipping duration",
        "Invalid or missing scheduled shipping duration"
    ],
    default="Other invalid duration"
)

ineligible_records.to_csv(
    INELIGIBLE_REPORT,
    index=False
)

print(f"Invalid-duration records saved: {len(ineligible_records):,}")
print(f"Cancelled shipments saved separately: {len(cancelled_records):,}")

# ------------------------------------------------------------
# GENERATE DELIVERY CLASSIFICATION SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("GENERATE DELIVERY CLASSIFICATION SUMMARY")
print("=" * 100)

classification_order = [
    "Delayed",
    "On-time",
    "Early",
    "Cancelled",
    "Ineligible"
]

classification_report = pd.DataFrame({
    "Delivery Classification": classification_order,
    "Record Count": [
        int(
            (df["Delivery Classification"] == category).sum()
        )
        for category in classification_order
    ]
})

classification_report["Percentage of All Records"] = (
    classification_report["Record Count"]
    / max(len(df), 1) * 100
)

classification_report["Percentage of Eligible Records"] = (
    classification_report["Record Count"]
    / max(eligible_count, 1) * 100
)

classification_report.loc[
    classification_report["Delivery Classification"].isin(
        ["Cancelled", "Ineligible"]
    ),
    "Percentage of Eligible Records"
] = np.nan

classification_report.to_csv(
    CLASSIFICATION_REPORT,
    index=False
)

print(classification_report.to_string(index=False))

# ------------------------------------------------------------
# GENERATE DELIVERY CLASSIFICATION BY SHIPPING MODE
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("DELIVERY CLASSIFICATION BY SHIPPING MODE")
print("=" * 100)

shipping_mode_summary = (
    df.loc[eligible_mask]
    .groupby(
        [MODE_COL, "Delivery Classification"],
        dropna=False
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for category in ["Delayed", "On-time", "Early"]:
    if category not in shipping_mode_summary.columns:
        shipping_mode_summary[category] = 0

shipping_mode_summary["Total Eligible Shipments"] = (
    shipping_mode_summary[
        ["Delayed", "On-time", "Early"]
    ].sum(axis=1)
)

shipping_mode_summary["Delayed Percentage"] = (
    shipping_mode_summary["Delayed"]
    / shipping_mode_summary[
        "Total Eligible Shipments"
    ].replace(0, np.nan)
    * 100
)

shipping_mode_summary.to_csv(
    TABLE_DIR / "delivery_classification_by_shipping_mode.csv",
    index=False
)

print(shipping_mode_summary.to_string(index=False))

# ------------------------------------------------------------
# GENERATE DELIVERY CLASSIFICATION BY REGION AND MARKET
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("DELIVERY CLASSIFICATION BY REGION AND MARKET")
print("=" * 100)

geographic_columns = [
    column
    for column in ["Order Region", "Market"]
    if column in df.columns
]

geographic_output_files = []

for geographic_column in geographic_columns:

    geographic_summary = (
        df.loc[eligible_mask]
        .groupby(
            [geographic_column, "Delivery Classification"],
            dropna=False
        )
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    for category in ["Delayed", "On-time", "Early"]:
        if category not in geographic_summary.columns:
            geographic_summary[category] = 0

    geographic_summary["Total Eligible Shipments"] = (
        geographic_summary[
            ["Delayed", "On-time", "Early"]
        ].sum(axis=1)
    )

    geographic_summary["Delayed Percentage"] = (
        geographic_summary["Delayed"]
        / geographic_summary[
            "Total Eligible Shipments"
        ].replace(0, np.nan)
        * 100
    )

    output_file = (
        TABLE_DIR
        / (
            "delivery_classification_by_"
            + geographic_column.lower().replace(" ", "_")
            + ".csv"
        )
    )

    geographic_summary.to_csv(
        output_file,
        index=False
    )

    geographic_output_files.append(output_file)

    print(f"\n{geographic_column} summary:")
    print(geographic_summary.to_string(index=False))

# ------------------------------------------------------------
# VALIDATE CLASSIFICATION TOTALS AND RECONCILIATION
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VALIDATE CLASSIFICATION TOTALS AND RECONCILIATION")
print("=" * 100)

classified_eligible_count = (
    delayed_count
    + on_time_count
    + early_count
)

all_classification_count = (
    delayed_count
    + on_time_count
    + early_count
    + cancelled_count
    + ineligible_count
)

validation_results = []

def add_validation(
    check_name,
    actual_value,
    expected_value,
    passed
):
    validation_results.append({
        "Validation Check": check_name,
        "Actual Value": str(actual_value),
        "Expected Value": str(expected_value),
        "Status": "PASS" if passed else "FAIL"
    })

add_validation(
    "Source record count preserved",
    len(df),
    original_row_count,
    len(df) == original_row_count
)

add_validation(
    "Delivery gap formula validation",
    "Correct" if gap_calculation_pass else "Incorrect",
    "Correct",
    gap_calculation_pass
)

add_validation(
    "Eligible classification totals reconcile",
    classified_eligible_count,
    int(eligible_mask.sum()),
    classified_eligible_count == int(eligible_mask.sum())
)

add_validation(
    "Cancelled records correctly separated",
    cancelled_count,
    int(cancelled_mask.sum()),
    cancelled_count == int(cancelled_mask.sum())
)

add_validation(
    "Invalid-duration records correctly separated",
    ineligible_count,
    int(ineligible_mask.sum()),
    ineligible_count == int(ineligible_mask.sum())
)

add_validation(
    "All classification totals reconcile",
    all_classification_count,
    len(df),
    all_classification_count == len(df)
)

add_validation(
    "No eligible record left unclassified",
    int(
        (
            df.loc[
                eligible_mask,
                "Delivery Classification"
            ] == "Not Classified"
        ).sum()
    ),
    0,
    not (
        df.loc[
            eligible_mask,
            "Delivery Classification"
        ] == "Not Classified"
    ).any()
)

add_validation(
    "No cancelled shipment classified incorrectly",
    int(
        (
            df.loc[
                cancelled_mask,
                "Delivery Classification"
            ] != "Cancelled"
        ).sum()
    ),
    0,
    (
        df.loc[
            cancelled_mask,
            "Delivery Classification"
        ] == "Cancelled"
    ).all()
)

add_validation(
    "No invalid-duration record classified incorrectly",
    int(
        (
            df.loc[
                ineligible_mask,
                "Delivery Classification"
            ] != "Ineligible"
        ).sum()
    ),
    0,
    (
        df.loc[
            ineligible_mask,
            "Delivery Classification"
        ] == "Ineligible"
    ).all()
)

add_validation(
    "No eligible delivery gap missing",
    int(
        df.loc[
            eligible_mask,
            "Delivery Delay Gap"
        ].isna().sum()
    ),
    0,
    df.loc[
        eligible_mask,
        "Delivery Delay Gap"
    ].notna().all()
)

validation_report = pd.DataFrame(
    validation_results
)

validation_report.to_csv(
    VALIDATION_REPORT,
    index=False
)

print(validation_report.to_string(index=False))

failed_checks = validation_report[
    validation_report["Status"] == "FAIL"
]

# ------------------------------------------------------------
# SAVE PROCESSED ANALYTICAL DATASET
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SAVE PROCESSED ANALYTICAL DATASET")
print("=" * 100)

df.to_csv(
    ANALYTICAL_FILE,
    index=False
)

saved_df = pd.read_csv(
    ANALYTICAL_FILE
)

analytical_file_check = (
    ANALYTICAL_FILE.exists()
    and len(saved_df) == len(df)
    and "Delivery Delay Gap" in saved_df.columns
    and "Delivery Classification" in saved_df.columns
)

print(f"Analytical dataset: {ANALYTICAL_FILE}")
print(f"Saved dataset shape: {saved_df.shape}")

print(
    f"Dataset saved successfully: "
    f"{'PASS' if analytical_file_check else 'FAIL'}"
)

# ------------------------------------------------------------
# SAVE DELIVERY GAP ANALYSIS SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SAVE DELIVERY GAP ANALYSIS SUMMARY")
print("=" * 100)

summary = {
    "analysis_name": "Delivery Gap Calculation and Classification",
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "source_file": str(INPUT_FILE),
    "analytical_file": str(ANALYTICAL_FILE),
    "source_records": int(original_row_count),
    "analytical_records": int(len(df)),
    "actual_duration_column": ACTUAL_COL,
    "scheduled_duration_column": SCHEDULED_COL,
    "formula": "Actual Shipping Days - Scheduled Shipping Days",
    "valid_duration_records": int(duration_valid.sum()),
    "invalid_duration_records": int(ineligible_count),
    "zero_scheduled_duration_records": int(zero_scheduled.sum()),
    "zero_scheduled_same_day_records": int(same_day_zero_count),
    "eligible_records": int(eligible_count),
    "delayed_records": int(delayed_count),
    "on_time_records": int(on_time_count),
    "early_records": int(early_count),
    "cancelled_records": int(cancelled_count),
    "delivery_status_mismatches": int(status_mismatch_mask.sum()),
    "risk_mismatches": int(risk_mismatch_mask.sum()),
    "eligible_records_with_invalid_risk": int(
        (eligible_mask & ~risk_valid).sum()
    ),
    "mean_delivery_gap": (
        float(valid_gaps.mean())
        if len(valid_gaps) else None
    ),
    "median_delivery_gap": (
        float(valid_gaps.median())
        if len(valid_gaps) else None
    ),
    "minimum_delivery_gap": (
        float(valid_gaps.min())
        if len(valid_gaps) else None
    ),
    "maximum_delivery_gap": (
        float(valid_gaps.max())
        if len(valid_gaps) else None
    ),
    "classification_reconciliation": bool(
        classified_eligible_count == eligible_count
    ),
    "all_records_reconciled": bool(
        all_classification_count == len(df)
    ),
    "validation_passed": bool(
        len(failed_checks) == 0
        and analytical_file_check
    ),
    "validation_checks_passed": int(
        (validation_report["Status"] == "PASS").sum()
    ),
    "validation_checks_failed": int(
        (validation_report["Status"] == "FAIL").sum()
    )
}

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        summary,
        file,
        indent=4,
        ensure_ascii=False
    )

print(f"Summary report: {SUMMARY_JSON}")

# ------------------------------------------------------------
# VERIFY ALL EXPECTED OUTPUT FILES
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("VERIFY ALL EXPECTED OUTPUT FILES")
print("=" * 100)

expected_files = [
    ANALYTICAL_FILE,
    CLASSIFICATION_REPORT,
    RISK_MISMATCH_REPORT,
    STATUS_MISMATCH_REPORT,
    CANCELLED_REPORT,
    INELIGIBLE_REPORT,
    ZERO_SCHEDULED_REPORT,
    VALIDATION_REPORT,
    SUMMARY_JSON,
    TABLE_DIR / "delivery_classification_by_shipping_mode.csv"
] + geographic_output_files

file_results = []

for file_path in expected_files:

    file_results.append({
        "Output File": str(
            file_path.relative_to(PROJECT_ROOT)
        ),
        "Status": (
            "PASS"
            if file_path.exists()
            else "FAIL"
        )
    })

file_report = pd.DataFrame(
    file_results
)

print(file_report.to_string(index=False))

all_files_saved = (
    file_report["Status"] == "PASS"
).all()

# ------------------------------------------------------------
# FINAL DELIVERY GAP ANALYSIS STATUS
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FINAL DELIVERY GAP ANALYSIS STATUS")
print("=" * 100)

print(f"Source records: {original_row_count:,}")
print(f"Analytical records: {len(df):,}")
print(f"Eligible records: {eligible_count:,}")
print(f"Delayed deliveries: {delayed_count:,}")
print(f"On-time deliveries: {on_time_count:,}")
print(f"Early deliveries: {early_count:,}")
print(f"Cancelled shipments: {cancelled_count:,}")
print(f"Invalid-duration records: {ineligible_count:,}")
print(f"Delivery Status mismatches: {status_mismatch_mask.sum():,}")
print(f"Late Delivery Risk mismatches: {risk_mismatch_mask.sum():,}")

print(
    f"Classification reconciliation: "
    f"{'PASS' if classified_eligible_count == eligible_count else 'FAIL'}"
)

print(
    f"All record reconciliation: "
    f"{'PASS' if all_classification_count == len(df) else 'FAIL'}"
)

print(
    f"Validation checks: "
    f"{(validation_report['Status'] == 'PASS').sum()} PASS / "
    f"{(validation_report['Status'] == 'FAIL').sum()} FAIL"
)

print(
    f"Output file verification: "
    f"{'PASS' if all_files_saved else 'FAIL'}"
)

if (
    len(failed_checks) == 0
    and analytical_file_check
    and all_files_saved
):
    print("\nDELIVERY GAP CALCULATION AND CLASSIFICATION COMPLETED SUCCESSFULLY")
    print("ALL VALIDATION CHECKS PASSED")
else:
    print("\nDELIVERY GAP ANALYSIS REQUIRES ATTENTION")

    if not failed_checks.empty:
        print("\nFailed validation checks:")
        print(failed_checks.to_string(index=False))

    if not all_files_saved:
        print("\nMissing output files:")
        print(
            file_report.loc[
                file_report["Status"] == "FAIL"
            ].to_string(index=False)
        )

DELIVERY GAP CALCULATION AND CLASSIFICATION

LOAD AND INSPECT CLEANED DATASET
Source file: C:\Users\hp\Analysis in Global Supply Chain Operations\data\processed\cleaned_logistics_data.csv
Dataset rows: 180,519
Dataset columns: 32
Actual shipping duration column: Days for shipping (real)
Scheduled shipping duration column: Days for shipment (scheduled)

IDENTIFY AND VALIDATE SHIPPING DURATIONS
Valid actual durations: 180,519
Invalid actual durations: 0
Valid scheduled durations: 180,519
Invalid scheduled durations: 0
Records with both valid durations: 180,519

CALCULATE DELIVERY DELAY GAP
Formula: Actual Shipping Days - Scheduled Shipping Days
Calculated valid gaps: 180,519
Minimum delivery gap: -2.00 days
Maximum delivery gap: 4.00 days
Average delivery gap: 0.57 days
Median delivery gap: 1.00 days
Gap calculation validation: PASS

INVESTIGATE ZERO SCHEDULED DURATION RECORDS
Zero scheduled duration records: 9,737

Zero scheduled duration by shipping mode:
Shipping Mode  Record Count
  